# Advanced Hybrid RAG Pipeline for Mutual Fund Factsheets
### Architecture: BM25 Lexical + Chroma Vector + FlashRank / Cross-Encoder Reranker + Qwen-2.5-8B-Instruct

This enhanced Colab notebook implements an end-to-end production RAG pipeline:
1. **Hybrid Parsing & Table Preservation:** PyMuPDF text & pdfplumber tabular markdown extraction with OCR fallback.
2. **Lexical Indexing (BM25):** Precise term-matching for financial tickers, ISINs, exact percentages, and scheme codes.
3. **Dense Vector Indexing (Chroma DB):** Semantic embeddings via `BAAI/bge-small-en-v1.5`.
4. **Hybrid Retrieval & Reranking:** Merges top candidates from BM25 and Chroma via Reciprocal Rank Fusion (RRF) / Cross-Encoder / FlashRank.
5. **Quantized LLM Engine:** `Qwen/Qwen2.5-8B-Instruct` loaded in 4-bit (`bitsandbytes` nf4) utilizing native ChatML formatting.
6. **Strict Grounded Guardrails:** Returns an explicit grounded fallback response with citation metadata if information cannot be verified in the retrieved factsheet.

## Step 1: Environment Setup & GPU Check

In [ ]:
# Verify NVIDIA GPU acceleration (Runs comfortably on Colab Free T4)
!nvidia-smi

# Install system dependencies for OCR and PDF rendering
!apt-get update -qq && apt-get install -y tesseract-ocr libtesseract-dev poppler-utils

# Install core Python packages
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    rank_bm25 \
    flashrank \
    chromadb \
    pymupdf \
    pdfplumber \
    pytesseract \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    torch

## Step 2: Document Parsing with Table & Text Structure Preservation
Mutual fund factsheets contain complex multi-column layouts, nested performance matrices, and raster charts. We parse tables into structured Markdown and fall back to OCR for low-text rasterized pages.

In [ ]:
import io
import os
from typing import List, Dict, Any
from PIL import Image
import fitz  # PyMuPDF
import pdfplumber
import pytesseract
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

class MutualFundFactsheetParser:
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path

    def parse_page(self, page_num: int) -> Dict[str, Any]:
        doc = fitz.open(self.pdf_path)
        page = doc[page_num]
        text = page.get_text("text").strip()
        ocr_used = False

        # OCR fallback if page has minimal raw text (e.g., scanned or graphic-heavy)
        if len(text) < 100:
            pix = page.get_pixmap(dpi=200)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img)
            ocr_used = True

        # Table extraction into formatted Markdown
        tables_md = []
        try:
            with pdfplumber.open(self.pdf_path) as pdf:
                plumber_page = pdf.pages[page_num]
                extracted_tables = plumber_page.extract_tables()
                for tbl in extracted_tables:
                    if tbl and len(tbl) > 1:
                        header = [str(c).replace('\n', ' ') if c else '' for c in tbl[0]]
                        rows = [[str(c).replace('\n', ' ') if c else '' for c in r] for r in tbl[1:]]
                        md = "| " + " | ".join(header) + " |\n"
                        md += "| " + " | ".join(['---'] * len(header)) + " |\n"
                        for r in rows:
                            md += "| " + " | ".join(r) + " |\n"
                        tables_md.append(md)
        except Exception as err:
            print(f"Page {page_num+1} table parsing warning: {err}")

        return {
            "page": page_num + 1,
            "text": text,
            "tables": tables_md,
            "ocr_used": ocr_used
        }

    def process_and_chunk(self, max_pages: int = 171) -> List[Document]:
        doc = fitz.open(self.pdf_path)
        total = min(len(doc), max_pages)
        raw_docs = []

        for p in range(total):
            res = self.parse_page(p)
            body = f"=== PAGE {res['page']} ===\n" + res['text'] + "\n\n"
            if res['tables']:
                body += "\n### TABLES DETECTED:\n" + "\n".join(res['tables'])
            raw_docs.append(
                Document(
                    page_content=body,
                    metadata={
                        "source": self.pdf_path,
                        "page": res["page"],
                        "ocr_used": res["ocr_used"],
                        "has_tables": len(res["tables"]) > 0
                    }
                )
            )

        # Chunking strategy keeping Markdown tables intact
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=150,
            separators=["\n\n### ", "\n| ", "\n\n", "\n", ". ", " "]
        )
        chunked_docs = splitter.split_documents(raw_docs)
        for idx, chk in enumerate(chunked_docs):
            chk.metadata["chunk_id"] = idx
        print(f"[Pipeline] Parsed {total} pages into {len(chunked_docs)} structured chunks.")
        return chunked_docs

## Step 3: Hybrid Indexing — BM25 + Chroma Vector DB
- **BM25Retriever:** Handles exact keyword matching (fund names, specific ticker percentages, expense ratios, ISIN codes).
- **Chroma Vector Store:** Handles conceptual & semantic queries (e.g., "Which funds provide downside protection in high-inflation regimes?").

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import torch

# Initialize dense embeddings
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

def setup_hybrid_indexes(chunks: List[Document]):
    print("[Indexing] Initializing BM25 Lexical Index...")
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 8  # Retrieve top 8 candidates

    print("[Indexing] Initializing Chroma Vector Index...")
    chroma_vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name="axis_factsheet_hybrid",
        persist_directory="./chroma_hybrid_db"
    )
    return bm25_retriever, chroma_vectorstore

## Step 4: Hybrid Retriever with Reranking (FlashRank / Cross-Encoder)
Standard ensemble retrievers may retrieve noisy documents. The Reranker evaluates full `(Query, Document)` pairs to score relevancy accurately before context injection.

In [ ]:
from flashrank import Ranker, RerankRequest

class HybridRerankRetriever:
    def __init__(self, bm25_retriever, vectorstore, top_k_candidates: int = 10, final_k: int = 4, score_threshold: float = 0.25):
        self.bm25 = bm25_retriever
        self.vectorstore = vectorstore
        self.top_k_candidates = top_k_candidates
        self.final_k = final_k
        self.score_threshold = score_threshold
        # Lightweight, high-precision cross-attention ranker
        self.ranker = Ranker(model_name="ms-marco-TinyBERT-L-2-v2", cache_dir="/root/.cache")

    def retrieve(self, query: str) -> List[Dict[str, Any]]:
        # 1. Fetch lexical candidates
        bm25_results = self.bm25.invoke(query)[:self.top_k_candidates]

        # 2. Fetch vector semantic candidates
        vector_results = self.vectorstore.similarity_search(query, k=self.top_k_candidates)

        # 3. Deduplicate across both retrieval channels
        candidate_map = {}
        for doc in bm25_results + vector_results:
            chunk_id = doc.metadata.get("chunk_id", doc.page_content[:50])
            if chunk_id not in candidate_map:
                candidate_map[chunk_id] = doc

        passages = [
            {"id": cid, "text": doc.page_content, "meta": doc.metadata}
            for cid, doc in candidate_map.items()
        ]

        if not passages:
            return []

        # 4. Rerank unified candidate pool
        rerank_request = RerankRequest(query=query, passages=passages)
        ranked_results = self.ranker.rerank(rerank_request)

        # 5. Filter by relevancy score threshold and select top final_k
        filtered_results = [r for r in ranked_results if r.get("score", 0.0) >= self.score_threshold]
        return filtered_results[:self.final_k]

## Step 5: Load Qwen-2.5-8B-Instruct with 4-bit Quantization
Loaded using `BitsAndBytesConfig` (`nf4` 4-bit) to consume ~5.5 GB VRAM on Colab T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-8B-Instruct"
print(f"[LLM] Loading {model_id} in 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,  # Low temperature for strict factual grounding
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)
print("[LLM] Qwen 8B loaded successfully.")

## Step 6: Native ChatML Prompt & Grounded Guardrails
If candidates fail the reranker score threshold, the pipeline intercepts execution and produces a strict grounded refusal rather than invoking the LLM to hallucinate.

In [ ]:
def generate_grounded_response(hybrid_retriever: HybridRerankRetriever, question: str) -> Dict[str, Any]:
    ranked_passages = hybrid_retriever.retrieve(question)

    # Grounded fallback condition: No documents retrieved or all candidates below relevancy threshold
    if not ranked_passages:
        return {
            "question": question,
            "answer": (
                "I could not find verified information answering your question in the provided "
                "Axis Mutual Fund factsheet. The query either refers to data not present in this document, "
                "or corresponds to visual elements (such as unlabelled Riskometer needles or chart arcs) "
                "that cannot be parsed with textual certainty."
            ),
            "status": "NOT_FOUND_GROUNDED",
            "sources": []
        }

    # Format retrieved passages with source citations
    context_parts = []
    source_pages = []
    for p in ranked_passages:
        pg = p.get("meta", {}).get("page", "Unknown")
        score = p.get("score", 0.0)
        source_pages.append(pg)
        context_parts.append(f"[Page {pg} | Relevancy Score: {score:.3f}]:\n{p.get('text')}")

    context_str = "\n\n---\n\n".join(context_parts)

    # Qwen native ChatML system and user messages
    system_prompt = (
        "You are a certified, meticulous mutual fund financial analyst. "
        "Answer the user's question accurately based SOLELY on the provided factsheet context. "
        "Rules to follow:\n"
        "1. Always cite exact page numbers for every financial figure or scheme metric mentioned.\n"
        "2. Do not assume or extrapolate figures not present in the text or tables.\n"
        "3. If the retrieved context does not contain enough specific data to answer completely, state: "
        "'The factsheet does not provide sufficient data to answer this conclusively.'"
    )

    user_prompt = f"Context Documents:\n{context_str}\n\nUser Query: {question}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    raw_answer = llm.invoke(formatted_input)

    return {
        "question": question,
        "answer": raw_answer,
        "status": "VERIFIED_ANSWER",
        "sources": sorted(list(set(source_pages)))
    }

## Step 7: End-to-End Pipeline Verification

In [ ]:
# Synthetic demonstration / Verification test
demo_chunks = [
    Document(
        page_content=(
            "Axis Large Cap Fund: Monthly Average AUM is 30,285.55 Cr, AUM as on 30th September, 2026 is 29,231.05 Cr. "
            "Benchmark: BSE 100 TRI. Top equity holdings include ICICI Bank Limited (9.29%), HDFC Bank Limited (8.25%), "
            "Bharti Airtel Limited (5.07%), Bajaj Finance Limited (4.94%). Exit Load: NIL."
        ),
        metadata={"page": 8, "chunk_id": 0}
    ),
    Document(
        page_content=(
            "Axis Midcap Fund: AUM as on 30th September, 2026 is 33,527.80 Cr. Benchmark: BSE Midcap 150 TRI. "
            "Top holdings: The Federal Bank Limited (3.98%), Apar Industries Limited (2.84%), Sona BLW (2.69%)."
        ),
        metadata={"page": 14, "chunk_id": 1}
    )
]

# Initialize indexes with demo documents (Replace with parser.process_and_chunk('Axis_Fund_Factsheet_September_2026.pdf'))
bm25_idx, chroma_idx = setup_hybrid_indexes(demo_chunks)
hybrid_retriever = HybridRerankRetriever(bm25_retriever=bm25_idx, vectorstore=chroma_idx, score_threshold=0.20)

# Query 1: Information Present
q1 = "What is the AUM and top holding of Axis Large Cap Fund as of September 2026?"
res1 = generate_grounded_response(hybrid_retriever, q1)
print("\n--- Query 1 Result ---")
print(f"Answer: {res1['answer']}")
print(f"Sources: {res1['sources']}")

# Query 2: Information Not Present (Triggers Grounded Fallback)
q2 = "What is the cryptocurrency allocation in Axis Balanced Advantage Fund?"
res2 = generate_grounded_response(hybrid_retriever, q2)
print("\n--- Query 2 Result (Fallback Guardrail) ---")
print(f"Status: {res2['status']}")
print(f"Answer: {res2['answer']}")